## BMLL Notebook Educational Series
### UK Market At Close: Analyzing Close-Time Liquidity Distribution

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
    # No dedicated sample views for these tables — fall back to production views
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"

### The Close-Price Execution Problem

**Challenge**: Execution algorithms often target the close auction or last-minute continuous trading
to benefit from increased liquidity and reduced execution risk. However, close-price liquidity:
- **Concentrates at specific venues** - Some venues have 90%+ of close volume
- **Varies dramatically by date** - Holiday schedules, market events affect distribution
- **Fragments across MTFs** - UK has 9+ trading venues competing for close business
- **Differs by security characteristics** - Liquid stocks have different patterns than illiquid ones

**The Risk**: Without understanding close-price liquidity distribution, algorithms may:
- Route orders to the wrong venues (lowest fill rates)
- Underestimate execution urgency (overestimate available liquidity)
- Miss arbitrage opportunities between venues at close

### Three Levels of Analysis

This notebook demonstrates three representations of increasing accuracy:

**1. Trade Counts** - Simple: How many trades occurred at close at each venue?
- ✓ Easy to compute and understand
- ✗ Ignores trade size (one large trade = many small trades)

**2. Trade Volume (Sum of Sizes)** - Better: How much volume was executed at each venue?
- ✓ Accounts for execution size
- ✗ Ignores the price at close (£100 × 1000 shares ≠ £200 × 500 shares)

**3. Trade Notional Value (Price × Size)** - Best: What was the economic value of close trading?
- ✓ Captures full economic significance
- ✓ Enables comparison across different securities
- ✓ Most relevant for portfolio managers (£ matters, not shares)

By analyzing all three, we get a complete picture of where close-price liquidity actually exists.

#### Import necessary libraries

In [ ]:
import pandas as pd

pd.options.display.max_columns = None
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

### Configuration: Venues and Time Period

**UK Trading Venues** (9 active venues for equities):
- **XLON**: London Stock Exchange (primary venue, ~80% of volume)
- **TRQX**: Turquoise (MTF, ~8% of volume)
- **AQXE**: Aquis Exchange (MTF, ~3% of volume)
- **BATE**: Cboe BZX Europe (MTF, ~2% of volume)
- **CHIX**: Cboe Europe (MTF, ~2% of volume)
- **SGMX**: Smartpool (dark venue)
- **XEQT**: Equiduct (MTF)
- **BOTC**: BOTC (off-exchange)
- **AQSE**: AQSE (formerly NEX - rebranded)

In [ ]:
uk_mics = ["XLON", "TRQX", "AQXE", "BATE", "CHIX", "SGMX", "XEQT", "BOTC", "AQSE"]

**Analysis Period**: Full calendar year 2025 (captures seasonal patterns, holiday effects)

In [ ]:
startd = "2025-01-01"
endd = "2025-12-31"

#### Get relevant securities only - Equities which are sterling-denominated and primary-listed on a UK MIC

## Data Retrieval Pipeline

The analysis requires:
1. **Reference data**: Security characteristics (ticker, currency, instrument type)
2. **Trades-Plus data**: Individual trade records with timestamp, venue, and price/size

We filter to:
- **Primary-listed UK securities** - Focus on securities with UK primary venues
- **Sterling-denominated** - GBp (pence) and GBP only
- **Equities only** - Exclude derivatives, bonds, etc.

This ensures we're analyzing a coherent dataset of UK-traded stocks.

In [ ]:
def _sql_list(values):
    """
    Convert Python list/set to SQL-formatted list string.

    Converts Python collections into the format needed for SQL IN clauses.
    Example: ['XLON', 'TRQX'] becomes "'XLON', 'TRQX'"

    Parameters:
        values: List, tuple, set, or single value to convert

    Returns:
        str: Comma-separated SQL list with quoted values (e.g., "'XLON', 'TRQX'")
    """
    if isinstance(values, (list, tuple, set)):
        items = list(values)
    else:
        items = [values]
    return ", ".join(f"'{v}'" for v in items)

In [ ]:

mic_list = _sql_list(uk_mics)

# ref is small enough to collect to pandas
ref_sql = f"""
    SELECT
        REFERENCE_DATE AS TradeDate,
        MIC,
        OPOL,
        CURRENCY_CODE AS CurrencyCode,
        INSTRUMENT_TYPE AS InstrumentType,
        IS_PRIMARY AS IsPrimary,
        LISTING_ID AS ListingId,
        INSTRUMENT_ID AS InstrumentId,
        TICKER AS Ticker
    FROM {REFERENCE_TBL}
    WHERE MIC IN ({mic_list})
      AND REFERENCE_DATE BETWEEN '{startd}' AND '{endd}'
"""
ref = spark.sql(ref_sql).toPandas()
ref = ref[
    ref["OPOL"].isin(uk_mics)
    & ref["CurrencyCode"].isin(["GBp", "GBP"])
    & ref["InstrumentType"].isin(["Equity"])
]

# trades-plus is too large to collect to the driver — keep as Spark temp views
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW v_ref AS
    SELECT LISTING_ID AS ListingId, REFERENCE_DATE AS TradeDate, IS_PRIMARY AS IsPrimary
    FROM {REFERENCE_TBL}
    WHERE MIC IN ({mic_list})
      AND REFERENCE_DATE BETWEEN '{startd}' AND '{endd}'
      AND OPOL IN ({mic_list})
      AND CURRENCY_CODE IN ('GBp', 'GBP')
      AND INSTRUMENT_TYPE = 'Equity'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW v_tp AS
    SELECT
        tp.TRADE_DATE AS TradeDate,
        tp.LISTING_ID AS ListingId,
        tp.INSTRUMENT_ID AS InstrumentId,
        tp.CLASSIFICATION AS Classification,
        tp.EXECUTION_VENUE AS ExecutionVenue,
        tp.TRADE_TIMESTAMP_NANOSECONDS AS TradeTimestampNanoseconds,
        tp.PRICE AS Price,
        tp.TRADING_MODE AS TradingMode,
        tp.BENCHMARK_INDICATOR AS BenchmarkIndicator,
        tp.PRICE_FORMATION AS PriceFormation,
        tp.TRADE_NOTIONAL_USD AS TradeNotionalUSD,
        tp.TICKER AS Ticker,
        r.IsPrimary
    FROM {TRADES_PLUS_TBL} tp
    INNER JOIN v_ref r ON r.ListingId = tp.LISTING_ID AND r.TradeDate = tp.TRADE_DATE
    WHERE tp.MIC IN ({mic_list})
      AND tp.TRADE_DATE BETWEEN '{startd}' AND '{endd}'
""")

spark.sql("""CREATE OR REPLACE TEMPORARY VIEW `v_closing_auc` AS SELECT `TradeDate`, `InstrumentId`, FIRST(`TradeTimestampNanoseconds`) AS `ClosingAuctionTimestamp`, FIRST(`Price`) AS `ClosingAuctionPrice` FROM `v_tp` WHERE `Classification` = 'LIT_CLOSING_AUCTION' AND `IsPrimary` = TRUE GROUP BY `TradeDate`, `InstrumentId`""")

spark.sql("""CREATE OR REPLACE TEMPORARY VIEW `v_tp_w_clse` AS SELECT `tp`.*, `ca`.`ClosingAuctionTimestamp`, `ca`.`ClosingAuctionPrice` FROM `v_tp` AS `tp` LEFT JOIN `v_closing_auc` AS `ca` ON `ca`.`TradeDate` = `tp`.`TradeDate` AND `ca`.`InstrumentId` = `tp`.`InstrumentId`""")

#### Get time/price of primary closing auction

#### Get closing auction notional as proportion of total lit notional

In [ ]:
tp_lit_daily_pd = spark.sql("""SELECT `TradeDate`, `Classification`, SUM(`TradeNotionalUSD`) AS `TradeNotionalUSD` FROM `v_tp` WHERE `Classification` IN ('LIT_CONTINUOUS', 'LIT_OPENING_AUCTION', 'LIT_CLOSING_AUCTION', 'LIT_INTRADAY_AUCTION', 'LIT_UNSCHEDULED_AUCTION', 'POST_TRADE_UNCROSSING', 'CLOSING_PRICE') GROUP BY `TradeDate`, `Classification`""").toPandas()
tp_lit_daily_pd["TradeDate"] = pd.to_datetime(tp_lit_daily_pd["TradeDate"])
tp_lit_daily_pd

#### Aggregate to monthly values

In [ ]:
df_lit = (
    tp_lit_daily_pd.groupby([pd.Grouper(key="TradeDate", freq="M"), "Classification"])[["TradeNotionalUSD"]]
    .sum()
    .reset_index()
)

df_lit_overall = (
    tp_lit_daily_pd.groupby([pd.Grouper(key="TradeDate", freq="M")])[["TradeNotionalUSD"]]
    .sum()
    .reset_index()
    .rename(columns={"TradeNotionalUSD": "TotalNotional"})
)
df_lit = df_lit.merge(df_lit_overall, on="TradeDate", how="left")
df_lit["Proportion"] = 100 * (df_lit["TradeNotionalUSD"] / df_lit["TotalNotional"])


df_lit["Month"] = df_lit["TradeDate"].dt.strftime("%b %Y")

In [ ]:
cdn_url = "https://cdn.plot.ly/plotly-2.14.0.min.js"

In [ ]:
df_close = df_lit[df_lit.Classification == "LIT_CLOSING_AUCTION"]

fig_lit = make_subplots(specs=[[{"secondary_y": True}]])

fig_lit = fig_lit.add_trace(
    go.Bar(
        name="Primary Closing Auction Notional (USD)",
        x=df_close.Month,
        y=df_close["TradeNotionalUSD"],
        marker_color="#146BC1",
    ),
    secondary_y=False,
)
fig_lit = fig_lit.add_trace(
    go.Scatter(
        name="Primary Closing Auction as Proportion of Lit Notional",
        x=df_close.Month,
        y=df_close["Proportion"],
        line=dict(color="#DE5F46"),
    ),
    secondary_y=True,
)


fig_lit.update_layout(template="plotly_white")
fig_lit.update_xaxes(title="Month")
fig_lit.update_yaxes(title_text="TradeNotionalUSD", secondary_y=False)
fig_lit.update_yaxes(title_text="Proportion of Lit Notional", secondary_y=True)


fig_lit.update_layout(bargap=0.4, bargroupgap=0.0)
fig_lit.update_xaxes(showgrid=False)
fig_lit.update_yaxes(showgrid=False)


fig_lit.update_layout(
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0)
)


fig_lit.show()

#### Get all trades executed within 10 mins of primary closing auction (irrespective of price)

In [ ]:
tp_all_mac_report_pd = spark.sql("""SELECT `TradeDate`, `Classification`, `ExecutionVenue`, SUM(`TradeNotionalUSD`) AS `TradeNotionalUSD` FROM `v_tp_w_clse` WHERE (`TradeTimestampNanoseconds` - `ClosingAuctionTimestamp`) BETWEEN 0 AND 600000000000 GROUP BY `TradeDate`, `Classification`, `ExecutionVenue`""").toPandas()
tp_all_mac_report_pd["TradeDate"] = pd.to_datetime(tp_all_mac_report_pd["TradeDate"])
tp_all_mac_report_pd

#### Roll up classifications to higher-level mechanisms

In [ ]:
tp_all_mac_report_pd.loc[
    (tp_all_mac_report_pd.Classification.isin(["BENCHMARK_PRICE", "SPECIAL_PRICE"]))
    & (tp_all_mac_report_pd.ExecutionVenue == "XOFF"),
    "Classification",
] = "OTC"
tp_all_mac_report_pd.loc[
    (tp_all_mac_report_pd.Classification.isin(["BENCHMARK_PRICE", "SPECIAL_PRICE"]))
    & (tp_all_mac_report_pd.ExecutionVenue != "XOFF"),
    "Classification",
] = "OFF_BOOK_ON_EXCHANGE"
tp_all_mac_report_pd.loc[
    (tp_all_mac_report_pd.ExecutionVenue == "SINT"), "Classification"
] = "SI"
tp_all_mac_report_pd.loc[
    (tp_all_mac_report_pd.Classification.str.contains("REQUEST_FOR_QUOTE_")),
    "Classification",
] = "RFQ"
tp_all_mac_report_pd.loc[
    (tp_all_mac_report_pd.Classification.str.contains("DARK_")), "Classification"
] = "DARK"

#### Aggregate and plot this

In [ ]:
df_all_report = (
    tp_all_mac_report_pd[
        ~tp_all_mac_report_pd.Classification.isin(["DARK", "AUCTION_ON_DEMAND", "RFQ"])
    ]
    .groupby([pd.Grouper(key="TradeDate", freq="M"), "Classification"])[["TradeNotionalUSD"]]
    .sum()
    .reset_index()
)
df_all_report["Month"] = df_all_report["TradeDate"].dt.strftime("%b %Y")

fig_all_mac_report = px.area(
    df_all_report,
    x="Month",
    y="TradeNotionalUSD",
    color="Classification",
    groupnorm="percent",
    template="plotly_white",
)
fig_all_mac_report.update_layout(legend_title_text="Mechanism")
fig_all_mac_report.update_layout(legend_traceorder="reversed")


fig_all_mac_report.show()

#### Find only price-forming trades printed at the closing auction price (and within 10 mins after the uncross)

In [ ]:
tp_pf_mac_grouped_pd = spark.sql("""SELECT `TradeDate`, `Classification`, `ExecutionVenue`, SUM(`TradeNotionalUSD`) AS `TradeNotionalUSD` FROM `v_tp_w_clse` WHERE NOT `PriceFormation` IN ('T', 'J') AND (`TradeTimestampNanoseconds` - `ClosingAuctionTimestamp`) BETWEEN 0 AND 600000000000 AND ABS(`Price` - `ClosingAuctionPrice`) < 1e-5 GROUP BY `TradeDate`, `Classification`, `ExecutionVenue`""").toPandas()
tp_pf_mac_grouped_pd["TradeDate"] = pd.to_datetime(tp_pf_mac_grouped_pd["TradeDate"])
tp_pf_mac_grouped_pd

In [ ]:
tp_pf_mac_grouped_pd.loc[
    (tp_pf_mac_grouped_pd.Classification.isin(["BENCHMARK_PRICE", "SPECIAL_PRICE"]))
    & (tp_pf_mac_grouped_pd.ExecutionVenue == "XOFF"),
    "Classification",
] = "OTC"
tp_pf_mac_grouped_pd.loc[
    (tp_pf_mac_grouped_pd.Classification.isin(["BENCHMARK_PRICE", "SPECIAL_PRICE"]))
    & (tp_pf_mac_grouped_pd.ExecutionVenue != "XOFF"),
    "Classification",
] = "OFF_BOOK_ON_EXCHANGE"
tp_pf_mac_grouped_pd.loc[
    (tp_pf_mac_grouped_pd.ExecutionVenue == "SINT"), "Classification"
] = "SI"
tp_pf_mac_grouped_pd.loc[
    (tp_pf_mac_grouped_pd.Classification.str.contains("REQUEST_FOR_QUOTE_")),
    "Classification",
] = "RFQ"
tp_pf_mac_grouped_pd.loc[
    (tp_pf_mac_grouped_pd.Classification.str.contains("DARK_")), "Classification"
] = "DARK"

In [ ]:
df_pf_mac = (
    tp_pf_mac_grouped_pd[
        ~tp_pf_mac_grouped_pd.Classification.isin(["DARK", "AUCTION_ON_DEMAND", "RFQ"])
    ]
    .groupby([pd.Grouper(key="TradeDate", freq="M"), "Classification"])[["TradeNotionalUSD"]]
    .sum()
    .reset_index()
)
df_pf_mac["Month"] = df_pf_mac["TradeDate"].dt.strftime("%b %Y")

fig_pf_mac = px.area(
    df_pf_mac,
    x="Month",
    y="TradeNotionalUSD",
    color="Classification",
    groupnorm="percent",
    template="plotly_white",
)

fig_pf_mac.update_layout(legend_title_text="Mechanism")
fig_pf_mac.update_layout(legend_traceorder="reversed")

fig_pf_mac.show()

#### Find price-forming trades printed at the closing price and with the TradingMode/CLSE flag (at any time)

In [ ]:
tp_pf_clse_pd = spark.sql("""SELECT `TradeDate`, `Classification`, `ExecutionVenue`, SUM(`TradeNotionalUSD`) AS `TradeNotionalUSD` FROM `v_tp_w_clse` WHERE `Classification` IN ('LIT_CLOSING_AUCTION', 'POST_TRADE_UNCROSSING', 'CLOSING_PRICE') OR (NOT `PriceFormation` IN ('T', 'J') AND ABS(`Price` - `ClosingAuctionPrice`) < 1e-5 AND (`TradingMode` = '3' OR `BenchmarkIndicator` IN ('1', '2', '3', '4'))) GROUP BY `TradeDate`, `Classification`, `ExecutionVenue`""").toPandas()
tp_pf_clse_pd["TradeDate"] = pd.to_datetime(tp_pf_clse_pd["TradeDate"])
tp_pf_clse_pd

In [ ]:
tp_pf_clse_pd.loc[
    (tp_pf_clse_pd.Classification.isin(["BENCHMARK_PRICE", "SPECIAL_PRICE"]))
    & (tp_pf_clse_pd.ExecutionVenue == "XOFF"),
    "Classification",
] = "OTC"
tp_pf_clse_pd.loc[
    (tp_pf_clse_pd.Classification.isin(["BENCHMARK_PRICE", "SPECIAL_PRICE"]))
    & (tp_pf_clse_pd.ExecutionVenue != "XOFF"),
    "Classification",
] = "OFF_BOOK_ON_EXCHANGE"
tp_pf_clse_pd.loc[(tp_pf_clse_pd.ExecutionVenue == "SINT"), "Classification"] = "SI"
tp_pf_clse_pd.loc[
    (tp_pf_clse_pd.Classification.str.contains("REQUEST_FOR_QUOTE_")), "Classification"
] = "RFQ"
tp_pf_clse_pd.loc[
    (tp_pf_clse_pd.Classification.str.contains("DARK_")), "Classification"
] = "DARK"

In [ ]:
df_pf_clse = (
    tp_pf_clse_pd[
        ~tp_pf_clse_pd.Classification.isin(["DARK", "AUCTION_ON_DEMAND", "RFQ"])
    ]
    .groupby([pd.Grouper(key="TradeDate", freq="M"), "Classification"])[["TradeNotionalUSD"]]
    .sum()
    .reset_index()
)
df_pf_clse["Month"] = df_pf_clse["TradeDate"].dt.strftime("%b %Y")

fig_pf_clse = px.area(
    df_pf_clse,
    x="Month",
    y="TradeNotionalUSD",
    color="Classification",
    groupnorm="percent",
    template="plotly_white",
)

fig_pf_clse.update_layout(legend_title_text="Mechanism")
fig_pf_clse.update_layout(legend_traceorder="reversed")

fig_pf_clse.show()